# 03 — MapReduce: Map, Shuffle e Reduce

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 03/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Materiais complementares

- [Guia Técnico](./reference_materials/03_mapreduce_fundamentos.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_mapreduce_fundamentos.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
Implementar map/reduce local para explicar o fluxo original do Hadoop.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
Particionamento por chave, etapas da agregação e custo de movimentação de dados.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
from collections import defaultdict
texto='spark processa dados e hadoop processa dados em cluster'
pares=[(palavra,1) for palavra in texto.split()]
agrupados=defaultdict(list)
for chave,valor in pares: agrupados[chave].append(valor)
resultado={chave:sum(valores) for chave,valores in agrupados.items()}
print(resultado)

{'spark': 1, 'processa': 2, 'dados': 2, 'e': 1, 'hadoop': 1, 'em': 1, 'cluster': 1}


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
# Verificação da equivalência com uma implementação de referência.
from collections import Counter
assert resultado == dict(Counter(texto.split()))
print('WordCount validado; em produção o shuffle distribui pares por chave.')

WordCount validado; em produção o shuffle distribui pares por chave.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 03 reuniu conceitos e verificações de **combiner e equivalência de agregação**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** shuffle e combinador. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **shuffle e combinador** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — MapReduce em memória com combinador explícito

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
from collections import Counter
particoes = ['spark scala spark', 'python spark', 'python']
# Combiner local: soma por chave antes da etapa de redução global.
parciais = [Counter(parte.split()) for parte in particoes]
reduzido = Counter()
for parcial in parciais:
    reduzido.update(parcial)
referencia = Counter(' '.join(particoes).split())
assert reduzido == referencia
pares_sem_combiner = sum(len(parte.split()) for parte in particoes)
pares_com_combiner = sum(len(c) for c in parciais)
print('Contagens:', dict(sorted(reduzido.items())))
print('Pares antes/depois do combiner:', pares_sem_combiner, pares_com_combiner)
print('Redução de pares é conceitual; não mede tráfego de rede real.')


Contagens: {'python': 2, 'scala': 1, 'spark': 3}
Pares antes/depois do combiner: 6 5
Redução de pares é conceitual; não mede tráfego de rede real.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **shuffle e combinador** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
